# Intro 11 — Hypothesis Testing

Practice notebook for the **"Hypothesis Testing"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Basic concepts: null, alternative, p-value, \(\alpha\)

A **hypothesis test** is a formal procedure for deciding whether data support or
contradict a claim about a population parameter.

- **Null hypothesis** \(H_0\): a baseline claim (e.g. \(\mu = 0\)).
- **Alternative hypothesis** \(H_1\): what we consider if \(H_0\) seems
  implausible (e.g. \(\mu > 0\), or \(\mu \ne \mu_0\) for a two-sided test).

We use a **test statistic** (a function of the data) and compute a **p-value**:
the probability, under \(H_0\), of seeing a result *as extreme or more extreme*
than the observed one. If the p-value is below a chosen significance level
\(\alpha\), we **reject \(H_0\)**.

$$
\text{p-value} = P_{H_0}\!\big(\text{test stat at least as extreme as observed}\big).
$$

Two errors are possible:

| Decision      | \(H_0\) true               | \(H_1\) true              |
|---------------|----------------------------|----------------------------|
| Fail to reject| Correct                    | Type II error (\(\beta\))  |
| Reject \(H_0\)| Type I error (\(\alpha\))  | Correct (power \(=1-\beta\))|

The **Type I error rate** is \(P(\text{reject } H_0 \mid H_0\text{ true})\); we
design the test to keep it at most \(\alpha\). The **power** is
\(P(\text{reject } H_0 \mid H_1\text{ true})\).

Below we run the PDF manufacturing example as a two-sided z-test and verify the
decision rule by hand and with `scipy.stats`.


In [ ]:
# PDF example: mu0 = 10 mm, known sigma = 0.2 mm, n = 50, xbar = 10.05 mm
mu0 = 10.0
sigma = 0.2
n = 50
xbar = 10.05
alpha = 0.05

# Test statistic Z = (xbar - mu0) / (sigma / sqrt(n))
se = sigma / np.sqrt(n)
Z = (xbar - mu0) / se
print(f"se = sigma/sqrt(n) = {se:.6f}  (expected ~0.0283)")
print(f"Z = (xbar - mu0)/se = {Z:.6f}  (expected ~1.7678)")

# Critical value for a two-sided test at level alpha
z_crit = stats.norm.ppf(1 - alpha / 2)
print(f"z_{{0.025}} = {z_crit:.6f}  (expected ~1.959964)")
print(f"|Z| = {abs(Z):.4f} vs z_crit = {z_crit:.4f}")
print(f"Reject H0? {abs(Z) > z_crit}  (expected False)")

# Two-sided p-value: P(|N(0,1)| >= |Z|)
p_value = 2 * (1 - stats.norm.cdf(abs(Z)))
print(f"two-sided p-value (scratch) = {p_value:.6f}")

# --- via scipy.stats: one-sample z-test is not built-in, use norm directly ---
# Equivalently use stats.norm.sf (survival function = 1 - cdf)
p_value2 = 2 * stats.norm.sf(abs(Z))
print(f"two-sided p-value (scipy sf) = {p_value2:.6f}")

assert np.isclose(se, 0.2/np.sqrt(50), atol=1e-12)
assert np.isclose(Z, 1.7678, atol=1e-3)
assert np.isclose(z_crit, 1.959964, atol=1e-5)
assert np.isclose(p_value, p_value2)
print("Matches PDF example: do NOT reject H0 at 5% level ✔")


---
## Part 2 — Z-test for a mean (known variance)

Assume \(X_1,\dots,X_n\) are i.i.d. \(N(\mu,\sigma^2)\) with **known** \(\sigma^2\).
We test

$$
H_0: \mu = \mu_0 \quad \text{vs.} \quad H_1: \mu \ne \mu_0.
$$

The test statistic

$$
Z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}}
$$

has, under \(H_0\), a standard normal distribution \(N(0,1)\). For a two-sided
test at level \(\alpha\), we **reject** \(H_0\) if \(|Z| > z_{\alpha/2}\).

We now **simulate** the test many times *under the null* (data drawn from
\(\mu = \mu_0\)) and confirm the empirical Type I error rate equals the nominal
\(\alpha\).


In [ ]:
# Simulate the two-sided z-test under H0: mu = mu0 (true)
mu0 = 10.0
sigma = 0.2
n = 50
alpha = 0.05
B = 50_000

rng = np.random.default_rng(42)
# Draw B samples of size n from N(mu0, sigma^2) -- H0 is TRUE
samples = rng.normal(mu0, sigma, size=(B, n))
xbars = samples.mean(axis=1)
Z = (xbars - mu0) / (sigma / np.sqrt(n))

z_crit = stats.norm.ppf(1 - alpha / 2)
reject = np.abs(Z) > z_crit
type_I_rate = reject.mean()

# p-values under H0
p_vals = 2 * stats.norm.sf(np.abs(Z))

print(f"Nominal alpha        = {alpha:.4f}")
print(f"Empirical Type I rate = {type_I_rate:.4f}")
print(f"z_crit (two-sided 5%) = {z_crit:.6f}")
print(f"Mean p-value under H0 = {p_vals.mean():.4f}  (expected ~0.5)")

assert abs(type_I_rate - alpha) < 0.01, "Type I rate should be ~alpha"
print("Empirical Type I rate matches nominal alpha ✔")


---
## Part 3 — t-test for a mean (unknown variance)

If \(\sigma\) is **unknown**, we use the sample standard deviation \(s\) and the
statistic

$$
T = \frac{\bar{X} - \mu_0}{s/\sqrt{n}},
$$

which, under \(H_0\), follows a **Student's t** distribution with \(n-1\)
degrees of freedom. The logic is the same: large \(|T|\) suggests the sample
mean is too far from \(\mu_0\) to be explained by random sampling alone. For a
two-sided test at level \(\alpha\), reject if \(|T| > t_{\alpha/2,\,n-1}\).

`scipy.stats.ttest_1samp` does this directly. We run it on the PDF z-test data
(treating \(\sigma\) as unknown) and also verify the simulated Type I error
rate of the t-test under the null.


In [ ]:
# One-sample t-test on the PDF example data (sigma unknown)
mu0 = 10.0
n = 50
xbar = 10.05
# Reconstruct a sample with exactly this mean and a plausible s.
rng = np.random.default_rng(123)
sample = rng.normal(mu0, 0.2, size=n)
# Force the sample mean to match the PDF's xbar = 10.05
sample = sample + (xbar - sample.mean())

# t-test from scratch
s = sample.std(ddof=1)
T = (sample.mean() - mu0) / (s / np.sqrt(n))
df = n - 1
t_crit = stats.t.ppf(1 - alpha / 2, df)
p_scratch = 2 * stats.t.sf(abs(T), df)
print(f"Sample mean  = {sample.mean():.6f}  (expected 10.05)")
print(f"Sample sd s  = {s:.6f}")
print(f"T statistic  = {T:.6f}")
print(f"t_crit (49 df, 5%) = {t_crit:.6f}")
print(f"Two-sided p-value (scratch) = {p_scratch:.6f}")
print(f"Reject H0? {abs(T) > t_crit}")

# --- via scipy.stats.ttest_1samp ---
res = stats.ttest_1samp(sample, popmean=mu0)
print(f"scipy ttest_1samp: T={res.statistic:.6f}, p={res.pvalue:.6f}")

assert np.isclose(res.statistic, T)
assert np.isclose(res.pvalue, p_scratch)
print("From-scratch t-test matches scipy.stats.ttest_1samp ✔")


In [ ]:
# Simulate the two-sided t-test under H0 and confirm the Type I rate == alpha
mu0 = 3.0
sigma_true = 1.7
n = 25
alpha = 0.05
B = 50_000

rng = np.random.default_rng(0)
samples = rng.normal(mu0, sigma_true, size=(B, n))
xbars = samples.mean(axis=1)
ss = samples.std(axis=1, ddof=1)
T = (xbars - mu0) / (ss / np.sqrt(n))
df = n - 1
t_crit = stats.t.ppf(1 - alpha / 2, df)
reject = np.abs(T) > t_crit
type_I_rate = reject.mean()
p_vals = 2 * stats.t.sf(np.abs(T), df)

print(f"Nominal alpha        = {alpha:.4f}")
print(f"Empirical Type I rate = {type_I_rate:.4f}")
print(f"t_crit (df={df}, 5%)  = {t_crit:.6f}")
print(f"Mean p-value under H0 = {p_vals.mean():.4f}  (expected ~0.5)")
assert abs(type_I_rate - alpha) < 0.01
print("t-test Type I rate matches nominal alpha ✔")


---
## Part 4 — p-value distribution under the null (uniform)

A key theoretical fact: **if \(H_0\) is true, the p-value is
Uniform\((0,1)\)**. That is why \(P(\text{p-value} \leq \alpha) = \alpha\): the
rejection rate is exactly the nominal level.

We already computed the p-values for the \(B\) simulations in Parts 2 and 3. We
now plot their histogram against the uniform density and the empirical CDF
against the diagonal \(y = x\).


In [ ]:
# Reuse the Part 3 t-test p-values (null true) and visualise
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: histogram vs Uniform(0,1) density (height 1)
axes[0].hist(p_vals, bins=40, density=True, color="steelblue",
            edgecolor="white", alpha=0.8, label="simulated p-values")
axes[0].axhline(1.0, color="crimson", lw=2, ls="--", label="Uniform(0,1) density")
axes[0].set_xlabel("p-value"); axes[0].set_ylabel("density")
axes[0].set_title(f"p-values under H0 (t-test, n={n}, B={B})")
axes[0].legend()

# Right: empirical CDF vs diagonal
ps = np.sort(p_vals)
ecdf = np.arange(1, ps.size + 1) / ps.size
axes[1].plot(ps, ecdf, color="seagreen", lw=2, label="empirical CDF")
axes[1].plot([0, 1], [0, 1], color="crimson", ls="--", lw=2, label="Uniform CDF")
axes[1].set_xlabel("p-value"); axes[1].set_ylabel("CDF")
axes[1].set_title("p-value CDF under H0 is uniform")
axes[1].legend()

plt.tight_layout(); plt.show()

# Numerical check: fraction of p-values <= alpha == alpha
for a in [0.01, 0.05, 0.10]:
    rate = (p_vals <= a).mean()
    print(f"P(p <= {a:.2f}) = {rate:.4f}  (expected ~{a:.2f})")
    assert abs(rate - a) < 0.012
print("p-value distribution is Uniform(0,1) under H0 ✔")


---
## Part 5 — Power vs effect size

**Power** is the probability of *correctly* rejecting \(H_0\) when \(H_1\) is
true, i.e. when the true mean \(\mu \ne \mu_0\). The further the true \(\mu\) is
from \(\mu_0\), the easier it is to detect the difference, so power grows with
the **effect size** \(\Delta = \mu - \mu_0\).

For the two-sided z-test with known \(\sigma\), the theoretical power at true
mean \(\mu\) is

$$
\pi(\mu) = P\!\left(|Z| > z_{\alpha/2} \;\middle|\; \mu\right),
\qquad
Z = \frac{\bar X - \mu_0}{\sigma/\sqrt{n}} \sim N\!\left(\frac{\mu-\mu_0}{\sigma/\sqrt{n}},\,1\right).
$$

We compute this exactly and compare with a Monte Carlo estimate, then sweep
effect size to plot the power curve.


In [ ]:
# Power of the two-sided z-test: theory vs simulation, as a function of effect
mu0 = 10.0
sigma = 0.2
n = 50
alpha = 0.05
z_crit = stats.norm.ppf(1 - alpha / 2)

deltas = np.linspace(-0.20, 0.20, 81)        # effect sizes mu - mu0
mus = mu0 + deltas

# Exact power: Z ~ N(delta/(sigma/sqrt(n)), 1) under the alternative
se = sigma / np.sqrt(n)
nc = deltas / se                              # non-centrality parameter
power_exact = (stats.norm.sf(z_crit, loc=nc) +
               stats.norm.cdf(-z_crit, loc=nc))

# Monte Carlo power at a few effect sizes
rng = np.random.default_rng(99)
B = 20_000
mc_effects = [-0.10, -0.05, -0.02, 0.0, 0.02, 0.05, 0.10]
power_mc = []
for mu in mc_effects:
    xbars = rng.normal(mu, sigma, size=(B, n)).mean(axis=1)
    Z = (xbars - mu0) / se
    power_mc.append((np.abs(Z) > z_crit).mean())

print(f"{'effect':>8} | {'exact power':>11} | {'MC power':>9}")
print("-" * 38)
for mu, pmc in zip(mc_effects, power_mc):
    idx = np.argmin(np.abs(deltas - (mu - mu0)))
    print(f"{mu - mu0:+8.3f} | {power_exact[idx]:11.4f} | {pmc:9.4f}")

# Sanity: zero effect => power == alpha
assert abs(power_exact[deltas.size // 2] - alpha) < 1e-6
# Sanity: MC matches exact within Monte Carlo error
for mu, pmc in zip(mc_effects, power_mc):
    idx = np.argmin(np.abs(deltas - (mu - mu0)))
    assert abs(power_exact[idx] - pmc) < 0.02
print("Exact and Monte Carlo power agree ✔")


In [ ]:
# Plot the power curve vs effect size
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(deltas, power_exact, color="crimson", lw=2, label="exact power")
ax.axhline(alpha, color="black", ls=":", lw=1, label=f"alpha = {alpha}")
ax.axvline(0, color="gray", ls="--", lw=1)
ax.scatter([m - mu0 for m in mc_effects], power_mc,
           color="steelblue", zorder=5, label="Monte Carlo")
ax.set_xlabel(r"effect size $\Delta = \mu - \mu_0$")
ax.set_ylabel("power")
ax.set_title(f"Two-sided z-test power (n={n}, sigma={sigma}, alpha={alpha})")
ax.legend()
plt.tight_layout(); plt.show()

# Show that power grows with n too, for a fixed effect
delta_fixed = 0.05
ns = np.array([10, 20, 50, 100, 200, 500, 1000])
power_n = stats.norm.sf(z_crit, loc=delta_fixed / (sigma / np.sqrt(ns))) + \
          stats.norm.cdf(-z_crit, loc=delta_fixed / (sigma / np.sqrt(ns)))
print(f"{'n':>6} | {'power (delta=0.05)':>18}")
for nn, pp in zip(ns, power_n):
    print(f"{nn:6d} | {pp:18.4f}")


---
**Your turn:**

- In Part 2, change the test to **one-sided** (\(H_1: \mu > \mu_0\)). What is the
  new critical value and the new Type I error rate? Confirm it is still
  \(\alpha\).
- In Part 3, reduce the sample size to \(n=5\). How does the t-test's Type I rate
  behave? (It should still be \(\approx \alpha\) — that is the point of using
  the t distribution instead of z.)
- In Part 4, repeat the p-value CDF check but with **non-normal** data
  (e.g. exponential). Is the p-value still uniform under \(H_0\) for the t-test
  when \(n\) is small? When \(n\) is large? (Think CLT.)
- In Part 5, fix the effect size and sweep \(n\). How large must \(n\) be to
  reach 80% power for a tiny effect \(\Delta = 0.02\)? For \(\Delta = 0.01\)?
- Recompute the Part 5 power curve at \(\alpha = 0.01\). Does the curve shift
  down everywhere? Why?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. A two-sided z-test uses \(n=64\), known \(\sigma=2.0\), \(\bar X=14.6\),
\(\mu_0=14.0\). Compute the test statistic, the 5% critical value, the p-value,
and state whether you reject \(H_0\).

2. A two-sided t-test uses \(n=16\), \(\bar X=72.5\), \(s=4.0\),
\(\mu_0=70.0\). Compute the t-statistic, the degrees of freedom, the 5%
critical value \(t_{0.025,\,15}\), the p-value, and state the decision.

3. Simulate the one-sided z-test \(H_0:\mu=0\) vs \(H_1:\mu>0\) with
\(n=40\), known \(\sigma=1\), \(\alpha=0.05\), using \(B=50\,000\) replicates
under the null (seed 42). Report the empirical Type I error rate and confirm it
is \(\approx 0.05\).

4. For a two-sided z-test with \(n=50\), \(\sigma=0.2\), \(\alpha=0.05\),
compute the exact power at true mean \(\mu = 10.1\) (i.e. effect size
\(\Delta=0.1\), \(\mu_0=10\)). Then estimate the same power by Monte Carlo with
\(B=20\,000\) and seed 7 and confirm the two agree.

5. Write a function `z_test(x, mu0, sigma, alpha=0.05, alternative="two-sided")`
    that returns `(Z, p_value, reject)`. Support `"two-sided"`, `"greater"`, and
    `"less"` alternatives. Verify it on the PDF example (\(\mu_0=10\),
    \(\sigma=0.2\), \(n=50\), \(\bar X=10.05\)): two-sided p-value
    \(\approx 0.077\), do not reject.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(Z = (14.6-14.0)/(2/\sqrt{64}) = 0.6/0.25 = 2.4\).
\(z_{0.025}\approx 1.96\). Two-sided p-value
\(= 2(1-\Phi(2.4)) \approx 0.0164\). Since \(|2.4|>1.96\) and p-value \(< 0.05\),
reject \(H_0\).

```python
n, sigma, xbar, mu0, alpha = 64, 2.0, 14.6, 14.0, 0.05
Z = (xbar - mu0) / (sigma/np.sqrt(n))
z_crit = stats.norm.ppf(1 - alpha/2)
p = 2 * stats.norm.sf(abs(Z))
print(Z, z_crit, p, abs(Z) > z_crit)  # 2.4 1.96 0.0164 True
```

**2.** d.f. \(=15\); \(t_{0.025,15}\approx 2.1314\).
\(T=(72.5-70.0)/(4/\sqrt{16})=2.5/1.0=2.5\).
Two-sided p-value \(= 2\,P(T_{15} > 2.5)\approx 0.0253\). Since \(|2.5|>2.1314\),
reject \(H_0\).

```python
n, xbar, s, mu0, alpha = 16, 72.5, 4.0, 70.0, 0.05
df = n - 1
T = (xbar - mu0) / (s/np.sqrt(n))
t_crit = stats.t.ppf(1 - alpha/2, df)
p = 2 * stats.t.sf(abs(T), df)
print(T, df, t_crit, p, abs(T) > t_crit)  # 2.5 15 2.1314 0.0253 True
```

**3.** One-sided test rejects when \(Z > z_{0.05}=1.6449\). Empirical Type I
rate should be \(\approx 0.05\).

```python
rng = np.random.default_rng(42)
n, sigma, mu0, alpha, B = 40, 1.0, 0.0, 0.05, 50_000
xbars = rng.normal(mu0, sigma, size=(B, n)).mean(axis=1)
Z = xbars / (sigma/np.sqrt(n))
z_crit = stats.norm.ppf(1 - alpha)          # one-sided
reject = Z > z_crit
print(reject.mean())                        # ~0.050
```

**4.** Exact power: non-centrality \(\delta = 0.1/(0.2/\sqrt{50}) \approx 3.5355\).
\(\pi = P(Z > 1.96 \mid \delta) + P(Z < -1.96 \mid \delta) \approx 0.4195\).
Monte Carlo with \(B=20\,000\), seed 7 gives \(\approx 0.42\).

```python
mu0, sigma, n, alpha = 10.0, 0.2, 50, 0.05
delta = 0.1
se = sigma/np.sqrt(n)
nc = delta/se
z_crit = stats.norm.ppf(1 - alpha/2)
power_exact = stats.norm.sf(z_crit, loc=nc) + stats.norm.cdf(-z_crit, loc=nc)
print(power_exact)                          # ~0.4195

rng = np.random.default_rng(7)
B = 20_000
xbars = rng.normal(mu0 + delta, sigma, size=(B, n)).mean(axis=1)
Z = (xbars - mu0)/se
print((np.abs(Z) > z_crit).mean())          # ~0.42
```

**5.** A reusable z-test supporting all three alternatives:

```python
def z_test(x, mu0, sigma, alpha=0.05, alternative="two-sided"):
    x = np.asarray(x, dtype=float)
    n = x.size
    Z = (x.mean() - mu0) / (sigma/np.sqrt(n))
    if alternative == "two-sided":
        p = 2 * stats.norm.sf(abs(Z))
        crit = stats.norm.ppf(1 - alpha/2)
        reject = abs(Z) > crit
    elif alternative == "greater":
        p = stats.norm.sf(Z)
        crit = stats.norm.ppf(1 - alpha)
        reject = Z > crit
    elif alternative == "less":
        p = stats.norm.cdf(Z)
        crit = stats.norm.ppf(alpha)
        reject = Z < crit
    else:
        raise ValueError(alternative)
    return Z, p, bool(reject)

# PDF example: n=50, xbar=10.05, mu0=10, sigma=0.2
x = np.full(50, 10.05)
Z, p, reject = z_test(x, 10.0, 0.2, alternative="two-sided")
print(Z, p, reject)  # 1.7678 0.0771 False
```


</details>
